In [ ]:
# ============================================================
# IMPROVED OSTEOPOROSIS ARCHITECTURE
# Directional Difference Texture CNN (DDT-CNN)
# ============================================================

# FIXES:
# ✅ Huge flatten vector → AdaptiveAvgPooling
# ✅ Pure ANN → CNN architecture
# ✅ Weak spatial learning → Convolutions
# ✅ No hierarchical features → Multi Conv Blocks
# ✅ No normalization → BatchNorm
# ✅ No texture enhancement → CLAHE
# ✅ No attention → SE Attention Block
# ✅ No pooling issue → Average Pooling only
#
# ============================================================

import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from PIL import Image

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# ============================================================
# SETTINGS
# ============================================================

DATASET_PATH = "total"

BATCH_SIZE = 32
EPOCHS = 10
IMG_SIZE = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# ============================================================
# CLAHE TRANSFORM
# ============================================================

class CLAHETransform:
    def __call__(self, img):

        img = np.array(img)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8,8)
        )

        img = clahe.apply(img)

        return Image.fromarray(img)

# ============================================================
# TRANSFORMS
# ============================================================

transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=1),

    # CLAHETransform(),

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize((0.5,), (0.5,))
])

# ============================================================
# DATASET
# ============================================================

dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=transform
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ============================================================
# DIRECTIONAL DIFFERENCE FEATURE EXTRACTION
# ============================================================

def get_features(x):

    center = x[:, :, 1:-1, 1:-1]

    n1 = x[:, :, :-2, :-2]
    n2 = x[:, :, :-2, 1:-1]
    n3 = x[:, :, :-2, 2:]

    n4 = x[:, :, 1:-1, :-2]
    n5 = x[:, :, 1:-1, 2:]

    n6 = x[:, :, 2:, :-2]
    n7 = x[:, :, 2:, 1:-1]
    n8 = x[:, :, 2:, 2:]

    features = torch.cat([

        torch.abs(center - n1),
        torch.abs(center - n2),
        torch.abs(center - n3),

        torch.abs(center - n4),
        torch.abs(center - n5),

        torch.abs(center - n6),
        torch.abs(center - n7),
        torch.abs(center - n8)

    ], dim=1)

    # ========================================================
    # NORMALIZATION OF DIFFERENCE MAPS
    # ========================================================

    features = (features - features.mean()) / (
        features.std() + 1e-6
    )

    return features

# ============================================================
# SE ATTENTION BLOCK
# ============================================================

class SEBlock(nn.Module):

    def __init__(self, channels, reduction=8):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(

            nn.Linear(channels, channels // reduction),

            nn.ReLU(),

            nn.Linear(channels // reduction, channels),

            nn.Sigmoid()
        )

    def forward(self, x):

        b, c, _, _ = x.size()

        y = self.pool(x).view(b, c)

        y = self.fc(y).view(b, c, 1, 1)

        return x * y

# ============================================================
# RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(channels),

            nn.ReLU(),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(channels)
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        residual = x

        out = self.block(x)

        out += residual

        out = self.relu(out)

        return out

# ============================================================
# IMPROVED MODEL
# ============================================================

class OsteoNetDD(nn.Module):

    def __init__(self):
        super().__init__()

        # ====================================================
        # SHALLOW TEXTURE CNN
        # ====================================================

        self.features = nn.Sequential(

            # -----------------------------------------------
            # Block 1
            # -----------------------------------------------

            nn.Conv2d(
                8,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.AvgPool2d(2),

            # -----------------------------------------------
            # Block 2
            # -----------------------------------------------

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            ResidualBlock(64),

            SEBlock(64),

            nn.AvgPool2d(2),

            # -----------------------------------------------
            # Block 3
            # -----------------------------------------------

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            ResidualBlock(128),

            SEBlock(128),

            # =================================================
            # GLOBAL AVERAGE POOLING
            # =================================================

            nn.AdaptiveAvgPool2d((1,1))
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.4),

            nn.Linear(128, 64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        # Directional Difference Maps
        x = get_features(x)

        # CNN Feature Learning
        x = self.features(x)

        # Classification
        x = self.classifier(x)

        return x

# ============================================================
# MODEL
# ============================================================

model = OsteoNetDD().to(device)

# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ============================================================
# TRAINING
# ============================================================

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {total_loss:.4f}"
    )

# ============================================================
# EVALUATION
# ============================================================

model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

        all_preds.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# ============================================================
# RESULTS
# ============================================================

accuracy = correct / total

print("\n===================================")
print("OSTEONET-DD RESULTS")
print("===================================")

print(f"\nAccuracy: {accuracy:.4f}")

print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")

cm = confusion_matrix(
    all_labels,
    all_preds
)

print(cm)

print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")

print(
    classification_report(
        all_labels,
        all_preds
    )
)

Using Device: cpu
Epoch [1/10] Loss: 42.2853
Epoch [2/10] Loss: 37.7800
Epoch [3/10] Loss: 37.5922
Epoch [4/10] Loss: 36.2196
Epoch [5/10] Loss: 36.1993
Epoch [6/10] Loss: 36.6453
Epoch [7/10] Loss: 36.4195
Epoch [8/10] Loss: 36.0998
Epoch [9/10] Loss: 35.2436
Epoch [10/10] Loss: 35.4481

OSTEONET-DD RESULTS

Accuracy: 0.8197

CONFUSION MATRIX
[[232 102]
 [ 13 291]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.95      0.69      0.80       334
           1       0.74      0.96      0.84       304

    accuracy                           0.82       638
   macro avg       0.84      0.83      0.82       638
weighted avg       0.85      0.82      0.82       638

